# Neural Network + FGM Evasion (CSV Pipeline)

This notebook runs a tabular anomaly-detection pipeline using:
- CSV input data
- PyTorch neural network
- ART Fast Gradient Method (FGM) evasion attack
- Optional adversarial training for robustness


In [1]:
# If needed, install once:
# !pip install torch scikit-learn adversarial-robustness-toolbox pandas numpy

import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DATA_PATH = r"CSVs\newDataset.csv"
LABEL_COL = "anomaly"

# Columns that are not model features in this dataset
DROP_COLS = {LABEL_COL, "segment", "train", "sampling"}

TEST_SIZE = 0.2
BATCH_SIZE = 128
NB_EPOCHS = 25
LR = 1e-3
FGM_EPS = 0.10


In [3]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_cols = [c for c in df.columns if c not in DROP_COLS]
    X = df[feature_cols].to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), feature_cols, scaler

X_train, X_test, y_train, y_test, feature_cols, scaler = load_and_prepare(DATA_PATH)


Loaded: CSVs\newDataset.csv
Rows=2123, Features=19, Label dist=[1689  434]
Train=(1698, 19), Test=(425, 19)


In [4]:
class MLP(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.net(x)


def make_art_classifier(d_in: int, lr: float = LR):
    model = MLP(d_in)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    return PyTorchClassifier(
        model=model,
        loss=criterion,
        optimizer=optimizer,
        input_shape=(d_in,),
        nb_classes=2,
        clip_values=(0.0, 1.0),
    )


In [5]:
def eval_classifier(art_clf: PyTorchClassifier, X: np.ndarray, y: np.ndarray, name: str):
    probs = art_clf.predict(X)
    y_pred = np.argmax(probs, axis=1)

    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y, y_pred))
    print(classification_report(y, y_pred, digits=4))

    return {"model_eval": name, "acc": acc, "f1": f1}


In [6]:
# 1) Train clean model
art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train, y_train, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

clean_on_clean = eval_classifier(art_clean, X_test, y_test, "clean_model_on_clean")

# 2) FGM evasion against clean model
fgm_clean = FastGradientMethod(estimator=art_clean, eps=FGM_EPS)
X_test_adv = fgm_clean.generate(x=X_test)

clean_on_adv = eval_classifier(art_clean, X_test_adv, y_test, "clean_model_on_fgm")



[clean_model_on_clean] acc=0.9224 f1=0.7925
confusion matrix:
[[329   9]
 [ 24  63]]
              precision    recall  f1-score   support

           0     0.9320    0.9734    0.9522       338
           1     0.8750    0.7241    0.7925        87

    accuracy                         0.9224       425
   macro avg     0.9035    0.8488    0.8723       425
weighted avg     0.9203    0.9224    0.9195       425


[clean_model_on_fgm] acc=0.1224 f1=0.1802
confusion matrix:
[[ 11 327]
 [ 46  41]]
              precision    recall  f1-score   support

           0     0.1930    0.0325    0.0557       338
           1     0.1114    0.4713    0.1802        87

    accuracy                         0.1224       425
   macro avg     0.1522    0.2519    0.1180       425
weighted avg     0.1763    0.1224    0.0812       425



In [7]:
# 3) Adversarial training with FGM
art_adv = make_art_classifier(d_in=X_train.shape[1])
fgm_for_training = FastGradientMethod(estimator=art_adv, eps=FGM_EPS)
trainer = AdversarialTrainer(classifier=art_adv, attacks=[fgm_for_training], ratio=0.5)
trainer.fit(X_train, y_train, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

art_adv = trainer.get_classifier()

adv_on_clean = eval_classifier(art_adv, X_test, y_test, "adv_trained_on_clean")
adv_on_adv = eval_classifier(art_adv, X_test_adv, y_test, "adv_trained_on_fgm")


Adversarial training epochs: 100%|██████████| 25/25 [00:01<00:00, 13.13it/s]


[adv_trained_on_clean] acc=0.8753 f1=0.5620
confusion matrix:
[[338   0]
 [ 53  34]]
              precision    recall  f1-score   support

           0     0.8645    1.0000    0.9273       338
           1     1.0000    0.3908    0.5620        87

    accuracy                         0.8753       425
   macro avg     0.9322    0.6954    0.7446       425
weighted avg     0.8922    0.8753    0.8525       425


[adv_trained_on_fgm] acc=0.8141 f1=0.2020
confusion matrix:
[[336   2]
 [ 77  10]]
              precision    recall  f1-score   support

           0     0.8136    0.9941    0.8948       338
           1     0.8333    0.1149    0.2020        87

    accuracy                         0.8141       425
   macro avg     0.8234    0.5545    0.5484       425
weighted avg     0.8176    0.8141    0.7530       425



In [8]:
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    adv_on_clean,
    adv_on_adv,
])
summary_df


,model_eval,acc,f1
0,clean_model_on_clean,0.922353,0.792453
1,clean_model_on_fgm,0.122353,0.180220
2,adv_trained_on_clean,0.875294,0.561983
3,adv_trained_on_fgm,0.814118,0.202020


In [9]:
# Optional: save metrics
out_csv = r"Results\NeuralNetworksResults\nn_fgm_evasion_summary.csv"
summary_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")


Saved: Results\NeuralNetworksResults\nn_fgm_evasion_summary.csv
